In [0]:
from pyspark.sql.functions import date_format, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("cod_municipio_rfb", StringType()),
    StructField("cod_municipio_ibge", StringType()),
    StructField("nome_municipio_rfb", StringType()),
    StructField("nome_municipio_ibge", StringType()),
    StructField("uf", StringType()),
])

(
    spark.read
        .option("sep", ";")
        .option("encoding", "iso-8859-1")
        .option("header", "true")
        .schema(schema)
        .csv("/Volumes/rfb/transient/transient/municipios")
    .withColumn("DATA_REF", date_format(current_timestamp(), "yyyy-MM"))
    .withColumn("ingested_at", current_timestamp())
    .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("rfb.raw.raw_municipios_rfb")
)

count = spark.table('rfb.raw.raw_municipios_rfb').where('DATA_REF == date_format(current_timestamp(), "yyyy-MM")').count()

if count > 0:
    print("Deleting transient folder.")
    print("Loading finish.")
    dbutils.fs.rm("/Volumes/rfb/transient/transient/municipios", recurse=True)
else:
    raise Exception("Raw table is empty, please check your source.")